# ⚡ Introduction to PySpark — Project Showcase

![PySpark](https://img.shields.io/badge/Library-PySpark-E25A1C?logo=apachespark&logoColor=white)
![Python](https://img.shields.io/badge/Language-Python-3776AB?logo=python&logoColor=white)
![Platform](https://img.shields.io/badge/Platform-DataCamp-05192D?logo=datacamp&logoColor=white)
![Status](https://img.shields.io/badge/Status-Completed-brightgreen)

> **Platform:** DataCamp | **Track:** Associate Data Engineer in Databricks (Course 11)
> **Instructor:** Benjamin Schmidt

**Datasets used in the exercises:** US Census adult groupings (`adult_reduced.csv`, `adults.json`), Data Scientist salaries (`salaries.csv`), and flights + destination airports.

**Objectives**
1. Understand how Spark distributes work and create a `SparkSession`.
2. Load CSV / JSON data into DataFrames, with inferred or explicit schemas.
3. Clean and transform DataFrames: nulls, new and renamed columns, joins, unions, complex types and UDFs.
4. Work with RDDs, query DataFrames with Spark SQL, aggregate, and inspect execution plans and caching.

The exercises ran in DataCamp's PySpark environment, where a `SparkSession` called `spark` and the datasets were preloaded, so the code cells below are transcriptions of my solutions and are not meant to be re-run here. Each result is shown with a cropped screenshot of the console output (`./screenshots/`).

| Chapter | Topic |
|---|---|
| 1 | Introduction to Apache Spark and PySpark |
| 2 | PySpark in Python — data manipulation and UDFs |
| 3 | Introduction to PySpark SQL — RDDs, Spark SQL, aggregations, performance |

## Chapter 1 — Introduction to Apache Spark and PySpark

### 1.1 — What PySpark is and when to use it

PySpark is the Python API for Apache Spark, a **distributed** processing engine built to handle datasets too large for one machine. It reads many formats (CSV, JSON, Parquet and more), lets me query data with both Python and SQL, and is optimized for speed at scale.

Typical use cases: big data analytics, distributed processing, real-time streaming, machine learning on large datasets, and **ETL / ELT pipelines**.

**Cluster roles:**
- The **master (driver) node** manages the cluster, schedules jobs and coordinates tasks.
- The **worker nodes** execute the tasks and store data in memory or on disk.

### 1.2 — SparkSession

The `SparkSession` is the entry point to the cluster: every DataFrame, SQL query and read goes through it. `.builder` sets it up, `.appName()` labels it (useful when several sessions exist), and `.getOrCreate()` returns the existing session or creates a new one — so the same line is safe to run twice.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("MySparkApp").getOrCreate()

**Exercise — creating a SparkSession.** Import `SparkSession`, create one named `my_spark` and print it to verify the object.

In [ ]:
# Import SparkSession from pyspark.sql
from pyspark.sql import SparkSession

# Create my_spark
my_spark = SparkSession.builder.appName("my_spark").getOrCreate()

# Print my_spark
print(my_spark)

**Result:** `<pyspark.sql.session.SparkSession object at 0x7fd35e246280>` — the object confirms an active session.

### 1.3 — PySpark DataFrames

A PySpark DataFrame is tabular data (rows and columns) with SQL-like operations — comparable to a pandas DataFrame or a SQL table, but distributed across the cluster. The basic workflow:

In [ ]:
# Read a CSV, using the first row as the header and letting Spark infer types
census_df = spark.read.csv("path/to/census.csv", header=True, inferSchema=True)

census_df.show(5)          # first rows
census_df.printSchema()    # column names, types and nullability

row_count = census_df.count()
print(f"Number of rows: {row_count}")

# SQL-like aggregation: average salary per gender (also sum, min, max)
census_df.groupBy("gender").agg({"salary_usd": "avg"}).show()

# Narrow the data: rows where age > 50, only two columns
census_df.filter(census_df["age"] > 50).select("age", "occupation").show()

The four functions I use most: `.select()` picks columns, `.filter()` keeps rows that match a condition, `.groupBy()` groups rows, and `.agg()` applies aggregate functions to each group.

**Exercise — loading census data.** Create a DataFrame from `adult_reduced.csv` with `spark.read.csv()` and show it.

In [ ]:
# Read in the CSV
census_adult = spark.read.csv('adult_reduced.csv')

# Show the DataFrame
census_adult.show()

![Without header=True, Spark names the columns _c0 … _c4 and treats the real header row (age, education.num, …) as data](./screenshots/ch1_ex2_census_read_no_header.png)

*Without header=True, Spark names the columns _c0 … _c4 and treats the real header row (age, education.num, …) as data*

**Interpretation:** this result is a useful lesson. Reading without options gives generic column names, keeps the header as the first data row, and loads everything as strings. `header=True` and `inferSchema=True` (or an explicit schema) fix all three issues.

**Exercise — reading a CSV and aggregating.** Load the salaries file with a header and inferred schema, count rows, and compare the average salary by company size.

In [ ]:
# Load the CSV file into a DataFrame
salaries_df = spark.read.csv("salaries.csv", header=True, inferSchema=True)

# Count the total number of rows
row_count = salaries_df.count()
print(f"Total rows: {row_count}")

# Group by company size and calculate the average of salaries
salaries_df.groupBy("company_size").agg({"salary_in_usd": "avg"}).show()
salaries_df.show()

![37,234 rows; average salary in USD by company size (L, M, S)](./screenshots/ch1_ex3_avg_salary_by_company_size.png)

*37,234 rows; average salary in USD by company size (L, M, S)*

![Preview of the salaries DataFrame with its inferred columns](./screenshots/ch1_ex3_salaries_preview.png)

*Preview of the salaries DataFrame with its inferred columns*

**Interpretation:** medium companies pay the most on average (~161.5K USD), large companies ~147.7K, and small companies clearly less (~88K). The gap is between small and the rest, not between medium and large.

**Exercise — filtering by company.** Average salary for **entry-level** jobs (`EN`) in **Canada** (`CA`).

In [ ]:
# Average salary for entry level in Canada
CA_jobs = ca_salaries_df.filter(ca_salaries_df['company_location'] == "CA").\
    filter(ca_salaries_df['experience_level'] == "EN").\
    groupBy().avg("salary_in_usd")

# Show the result
CA_jobs.show()

![Average entry-level salary in Canada](./screenshots/ch1_ex4_entry_level_canada_avg.png)

*Average entry-level salary in Canada*

**Interpretation:** about **97,331 USD**. Chaining two `.filter()` calls works like `AND`, and an empty `.groupBy()` aggregates the whole filtered DataFrame into a single row.

### 1.4 — More on Spark: data sources, schemas and data types

| Format | Use | Reader |
|---|---|---|
| CSV | structured, delimited data | `spark.read.csv("path/to/file.csv")` |
| JSON | semi-structured, hierarchical data | `spark.read.json("path/to/file.json")` |
| Parquet | columnar, optimized for storage and querying — common in data engineering | `spark.read.parquet("path/to/file.parquet")` |

Spark can **infer** a schema (`inferSchema=True`), but defining it **manually** gives better control for fixed data structures: no surprises in types, and no extra pass over the data to guess them.

| Type | Holds | Example |
|---|---|---|
| `IntegerType` | whole numbers | `1`, `3478`, `-1890456` |
| `LongType` | larger whole numbers (8-byte) | `922334775806` |
| `FloatType` / `DoubleType` | decimals | `3.14159` |
| `StringType` | text | `"This is an example of a string."` |
| `ArrayType`, `MapType`, `StructType` | lists, key-value pairs, nested records | see Chapter 2 |

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, ArrayType

schema = StructType([
    StructField("id", IntegerType(), True),        # True = nullable
    StructField("name", StringType(), True),
    StructField("scores", ArrayType(IntegerType()), True)
])

df = spark.createDataFrame(data, schema=schema)

**Basic DataFrame operations:** `.select()` chooses columns, `.filter()` / `.where()` filter rows, `.sort()` / `.orderBy()` order them, and `.na.drop()` removes rows with nulls.

In [ ]:
df.select("name", "age").show()
df.filter(df["age"] > 30).show()
df.where(df["age"] == 30).show()
df.sort("age", ascending=False).show()
df.na.drop().show()

**Exercise — infer and filter.** Load `adults.json` (schema inferred from the JSON) and keep adults over 40.

In [ ]:
# Load the dataframe
census_df = spark.read.json("adults.json")

# Filter rows based on age condition
salary_filtered_census = census_df.filter(census_df['age'] > 40)

# Show the result
salary_filtered_census.show()

![Adults over 40 read from JSON — column names come straight from the JSON keys](./screenshots/ch1_ex5_json_age_over_40.png)

*Adults over 40 read from JSON — column names come straight from the JSON keys*

**Interpretation:** unlike the header-less CSV, JSON carries field names, so the columns are named correctly with no extra options. Note that JSON fields come back in alphabetical order (`age`, `education.num`, `income`, …).

**Exercise — writing a schema.** Define the schema explicitly from the data dictionary and read `adult_reduced_100.csv` with it:

| Variable | Description |
|---|---|
| `age` | individual age |
| `education_num` | education by degree |
| `marital_status` | marital status |
| `occupation` | occupation |
| `income` | categorical income |

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Fill in the schema with the columns you need from the exercise instructions
schema = StructType([StructField("age", IntegerType()),
                     StructField("education_num", IntegerType()),
                     StructField("marital_status", StringType()),
                     StructField("occupation", StringType()),
                     StructField("income", StringType())
                    ])

# Read in the CSV, using the schema you defined above
census_adult = spark.read.csv("adult_reduced_100.csv", sep=',', header=False, schema=schema)

# Print out the schema
census_adult.printSchema()

![printSchema() shows exactly the names and types defined in the StructType](./screenshots/ch1_ex6_manual_schema.png)

*printSchema() shows exactly the names and types defined in the StructType*

**Interpretation:** `age` and `education_num` are integers, the rest strings — the types are guaranteed by the schema rather than guessed.

## Chapter 2 — PySpark in Python: Data Manipulation

### 2.1 — Handling missing data

In [ ]:
from pyspark.sql.functions import col

df_cleaned = df.na.drop()                                 # drop rows with any null
df_cleaned = df.where(col("columnName").isNotNull())      # drop nulls in one column only
df_filled  = df.na.fill({"age": 0})                       # replace nulls with a value

Dropping is simple but loses rows; filling keeps them with a default. The right choice depends on whether the missing value matters for the analysis.

**Exercise — dropping rows with nulls.**

In [ ]:
# Drop rows with any nulls
census_cleaned = census_df.na.drop()

# Show the result
census_cleaned.show()

![Rows that contained any null value are gone (e.g., the first rows with '?' entries are no longer the first rows)](./screenshots/ch2_ex1_drop_nulls.png)

*Rows that contained any null value are gone (e.g., the first rows with '?' entries are no longer the first rows)*

**Interpretation:** compared with the raw data, several rows disappeared (the 90- and 66-year-old rows are no longer at the top). This DataFrame was also read without a header, so the `_c0 … _c4` naming issue from Chapter 1 shows up again.

### 2.2 — Column and row operations

In [ ]:
df = df.withColumn("age_plus_5", df["age"] + 5)      # new column from a calculation
df = df.withColumnRenamed("age", "years")            # rename
df = df.drop("department")                           # remove a column

filtered_df = df.filter(df["salary"] > 50000)        # row filter
grouped_df = df.groupBy("department").avg("salary")  # group and summarize

**Exercise — creating and renaming columns.** Add `weekly_salary` (income / 52) and rename `age` to `years`.

In [ ]:
# Create a new column 'weekly_salary'
census_df_weekly = census_df.withColumn("weekly_salary", census_df.income / 52)

# Rename the 'age' column to 'years'
census_df_weekly = census_df_weekly.withColumnRenamed("age", "years")

# Show the result
census_df_weekly.show()

![New weekly_salary column and the age column renamed to years](./screenshots/ch2_ex2_weekly_salary_column.png)

*New weekly_salary column and the age column renamed to years*

**Interpretation:** a yearly income of 50,000 becomes ~961.54 per week. Each transformation returns a **new** DataFrame (DataFrames are immutable), which is why the result is reassigned.

### 2.3 — Advanced DataFrame operations: joins, unions and complex types

**Joins** combine rows on common columns, with the same types as SQL (`inner`, `left`, `right`, `outer`). **Union** stacks two DataFrames with the same schema.

In [ ]:
df_joined = df1.join(df2, on="id", how="inner")              # same key name
df_joined = df1.join(df2, df1.Id == df2.Name, "inner")       # different key names
df_union = df1.union(df2)                                    # identical schemas

**Complex types** keep related values inside one column:
- **Arrays** — lists, `ArrayType(StringType(), False)`
- **Maps** — key-value pairs, `MapType(StringType(), StringType())`
- **Structs** — nested records, `StructType([...])`

In [ ]:
from pyspark.sql.functions import array, struct, lit
from pyspark.sql.types import StructType, StructField, StringType, MapType

df = df.withColumn("scores", array(lit(85), lit(90), lit(78)))       # array column
df = df.withColumn("name_struct", struct("first_name", "last_name"))  # struct column

schema = StructType([
    StructField("name", StringType(), True),
    StructField("properties", MapType(StringType(), StringType()), True)   # map column
])

**Exercise — joining flights with their destination airports.** As a data engineer at a travel company, link each flight to its destination airport. The airports key is `faa`, so I renamed it to match the flights' `dest` column, then joined.

In [ ]:
# TODO: verificar — line 1 of this script is cut off in the screenshot
airports.show()
flights.show()
# .withColumnRenamed() renames the "faa" column to "dest"
airports = airports.withColumnRenamed("faa", "dest")

# Join the DataFrames
flights_with_airports = flights.join(airports, on='dest', how='leftouter')

# Examine the new DataFrame
flights_with_airports.show()

![Flights enriched with destination airport data, joined on dest](./screenshots/ch2_ex3_flights_with_airports.png)

*Flights enriched with destination airport data, joined on dest*

**Interpretation:** renaming the key first lets me use `on='dest'`, which keeps a single `dest` column in the output. A **left outer** join keeps every flight even if its destination isn't in the airports table, so no flight records are silently lost.

### 2.4 — User-defined functions (UDFs)

A **UDF** wraps custom Python logic so it can be applied to DataFrame columns and reused. There are two kinds:

| | PySpark UDF | pandas UDF |
|---|---|---|
| Best for | smaller datasets, simple cleaning | larger datasets, more complex operations |
| How it runs | row by row | vectorized, on pandas Series (batches) |
| Registration | must be registered with `udf()` | declared with the `@pandas_udf` decorator |

In [ ]:
from pyspark.sql.functions import udf, pandas_udf
from pyspark.sql.types import StringType

# PySpark UDF
def to_uppercase(s):
    return s.upper() if s else None

to_uppercase_udf = udf(to_uppercase, StringType())
df = df.withColumn("name_upper", to_uppercase_udf(df["name"]))

# pandas UDF
@pandas_udf("float")
def fahrenheit_to_celsius_pandas(temp_f):
    return (temp_f - 32) * 5.0 / 9.0

**Exercise — integers in PySpark UDFs.** Register the provided `age_category` function as a UDF and use it to add a `category` column.

In [ ]:
# Register the function age_category as a UDF
age_category_udf = udf(age_category, StringType())

# Apply your udf to the DataFrame
age_category_df_2 = age_category_df.withColumn("category", age_category_udf(age_category_df["age"]))

# Show df
age_category_df_2.show()

![Each person labelled Child, Adult or Senior by the UDF](./screenshots/ch2_ex4_age_category_udf.png)

*Each person labelled Child, Adult or Senior by the UDF*

**Interpretation:** the UDF replaces what would otherwise be a chain of `when()` conditions in a cleaning workflow, and it can be reused across DataFrames.

**Exercise — pandas UDFs.** Define `add_ten_pandas()` as a pandas UDF and apply it to the `value` column.

In [ ]:
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import DoubleType

# Define a Pandas UDF that adds 10 to each element in a vectorized way
@pandas_udf(DoubleType())
def add_ten_pandas(column):
    return column + 10

# Apply the UDF and show the result
df.show()
df = df.withColumn("10_plus", add_ten_pandas(df["value"]))
df.show()

![The 10_plus column adds 10 to every value in one vectorized operation](./screenshots/ch2_ex5_pandas_udf.png)

*The 10_plus column adds 10 to every value in one vectorized operation*

**Interpretation:** compared with the previous UDF, there's no `udf()` registration call — the decorator declares the return type, and `column + 10` runs on a whole pandas Series at once instead of one row at a time.

## Chapter 3 — Introduction to PySpark SQL

### 3.1 — Resilient Distributed Datasets (RDDs)

Spark **parallelizes** data and computation automatically: worker nodes process partitions in parallel and the results are combined at the end, which is what makes gigabytes or terabytes manageable.

An **RDD** is Spark's low-level distributed collection: immutable, spread across the cluster, and able to recover automatically from node failures. It is transformed with operations like `map()` and `filter()`, and results are brought back with actions like `collect()`.

| DataFrames | RDDs |
|---|---|
| high-level, optimized, easy to use | low-level, flexible but verbose |
| schema with named, typed columns (like a SQL table) | no schema — harder for structured data |
| SQL-like operations | weaker for analytics |

In [ ]:
census_rdd = census_df.rdd       # DataFrame -> RDD
census_rdd.collect()             # bring all records to the driver

for row in census_df.collect():  # collect() on a DataFrame returns a list of Row objects
    print(row)

`collect()` moves the **whole dataset** to the driver, so on real data it should be used only on small results.

**Exercise — creating RDDs.**

In [ ]:
# Create a DataFrame
df = spark.read.csv("salaries.csv", header=True, inferSchema=True)

# Convert DataFrame to RDD
rdd = df.rdd

# Show the RDD's contents
rdd.collect()
print(rdd)

**Result:** `MapPartitionsRDD[14] at javaToPython at NativeMethodAccessorImpl.java:0`

**Interpretation:** `print(rdd)` prints the RDD object, not its data — the RDD is a lazy description of the computation. To see the records, the output of `collect()` must be printed: `print(rdd.collect())`.

**Exercise — collecting RDDs and comparing with DataFrames.** Collect the salaries as an RDD, then get the maximum salary per experience level with the DataFrame API.

In [ ]:
# Create an RDD from the df_salaries
rdd_salaries = df_salaries.rdd
# Collect and print the results
print('rdd salaries: \n{}'.format(rdd_salaries.collect()))
# Group by the experience level and calculate the maximum salary
dataframe_results = df_salaries.groupBy("experience_level").agg({"salary_in_usd": 'max'})
# Show the results
dataframe_results.show()

![The RDD prints raw Row objects; the DataFrame gives a clean max salary per experience level](./screenshots/ch3_ex2_max_salary_by_experience.png)

*The RDD prints raw Row objects; the DataFrame gives a clean max salary per experience level*

**Interpretation:** the contrast is the point of the exercise — the RDD dump is hard to read, while one DataFrame line returns a tidy table. Mid-level (`MI`) has the highest single salary (450K), above senior (412K) and executive (325K), which suggests outliers rather than a clean progression.

### 3.2 — Spark SQL and temporary views

**Spark SQL** is the Spark module for structured data: it runs SQL queries alongside Python processing in the same application. A DataFrame becomes queryable after registering it as a **temporary view**, which exists only for the session and doesn't modify the underlying data.

In [ ]:
data = [("Alice", "HR", 30), ("Bob", "IT", 40), ("Cathy", "HR", 28)]
columns = ["Name", "Department", "Age"]
df = spark.createDataFrame(data, schema=columns)

df.createOrReplaceTempView("people")
result = spark.sql("SELECT Name, Age FROM people WHERE Age > 30")
result.show()

# SQL output is a DataFrame, so DataFrame methods can continue the pipeline
query_result = spark.sql("SELECT Name, Salary FROM employees WHERE Salary > 3000")
high_earners = query_result.withColumn("Bonus", query_result.Salary * 0.1)

**Exercise — querying a temp view.** Register `data_view` and compute total salary by position.

In [ ]:
# Register as a view
df.createOrReplaceTempView("data_view")

# Advanced SQL query: Calculate total salary by Position
result = spark.sql("""
    SELECT Position, SUM(Salary) AS Total_Salary
    FROM data_view
    GROUP BY Position
    ORDER BY Total_Salary DESC
    """
)
result.show()

![Total salary by position, highest first](./screenshots/ch3_ex3_total_salary_by_position.png)

*Total salary by position, highest first*

**Interpretation:** Engineers account for the largest total payroll (~2.54M), followed by Coordinators and Directors.

**Exercise — running SQL on DataFrames.**

In [ ]:
# Create a temporary table "people"
df.createOrReplaceTempView("people")

# Select the names from the temporary table people
query = """SELECT name FROM people"""

# Assign the result of Spark's query to people_df_names
people_df_names = spark.sql(query)

# Print the top 10 names of the people
people_df_names.show(10)

![Names selected from the people temp view](./screenshots/ch3_ex4_people_names.png)

*Names selected from the people temp view*

**Exercise — analytics with SQL on DataFrames.** Extract job titles in Canada and summarize them.

In [ ]:
# Create a temporary view of salaries_table
salaries_df.createOrReplaceTempView('salaries_table')

# Construct the "query"
query = '''SELECT job_title, salary_in_usd
FROM salaries_table
WHERE company_location == "CA"'''

# Apply the SQL "query"
canada_titles = spark.sql(query)

# Generate basic statistics
canada_titles.describe().show()

![describe() summary of Canadian job titles and salaries](./screenshots/ch3_ex5_canada_titles_describe.png)

*describe() summary of Canadian job titles and salaries*

**Interpretation:** 1,115 jobs in Canada with a mean of ~143K USD and a wide spread (std ~62K, range 15K–800K). `describe()` returns `null` mean and stddev for the text column, and min/max of strings are alphabetical — a reminder to read summary statistics by column type.

### 3.3 — PySpark aggregations

Common SQL aggregations work in `spark.sql()`, and SQL and DataFrame steps can be mixed freely:

In [ ]:
spark.sql("""
    SELECT Department, SUM(Salary) AS Total_Salary, AVG(Salary) AS Average
    FROM employees
    GROUP BY Department
""").show()

filtered_df = df.filter(df.Salary > 3000)                  # filter early with the DataFrame API
filtered_df.createOrReplaceTempView("filtered_employees")  # then aggregate with SQL

# Types matter: cast strings before aggregating
df = df.withColumn("Salary", df["Salary"].cast("int"))
df.groupBy("Department").sum("Salary").show()

# RDD equivalent: (key, value) pairs reduced by key
rdd = df.rdd.map(lambda row: (row["Department"], row["Salary"]))
rdd_aggregated = rdd.reduceByKey(lambda x, y: x + y)
print(rdd_aggregated.collect())

**Best practices:** filter early to reduce data before aggregating; make sure types are clean; minimize operations that touch the entire dataset (like broad `groupBy()`); prefer DataFrames over RDDs for their optimizations; and use `explain()` to check the execution plan.

**Exercise — aggregating in PySpark.** Minimum salary at small companies and maximum at large companies, filtering by referencing the column directly instead of a SQL string.

In [ ]:
# Find the minimum salaries for small companies
salaries_df.filter(salaries_df.company_size == "S").groupBy().min("salary_in_usd").show()

# Find the maximum salaries for large companies
salaries_df.filter(salaries_df.company_size == "L").groupBy().max("salary_in_usd").show()

![Minimum salary at small companies and maximum at large companies](./screenshots/ch3_ex6_min_max_salary_by_size.png)

*Minimum salary at small companies and maximum at large companies*

**Interpretation:** the minimum at small companies is 15,809 USD and the maximum at large companies is 423,000 USD. The instructions also mention US companies; my solution filters only by `company_size`, so restricting to the US would need an extra `.filter(salaries_df.company_location == "US")`.

**Exercise — aggregating in RDDs.**

In [ ]:
# DataFrame Creation
data = [("HR", "3000"), ("IT", "4000"), ("Finance", "3500")]
columns = ["Department", "Salary"]
df = spark.createDataFrame(data, schema=columns)

# Map the DataFrame to an RDD
rdd = df.rdd.map(lambda row: (row["Department"], row["Salary"]))

# Apply a lambda function to get the sum of the DataFrame
rdd_aggregated = rdd.reduceByKey(lambda x, y: x + y)

# Show the collected Results
print(rdd_aggregated.collect())

![reduceByKey output: one (department, salary) pair per key](./screenshots/ch3_ex7_rdd_reduce_by_key.png)

*reduceByKey output: one (department, salary) pair per key*

**Interpretation:** each department appears only once, so nothing is actually summed, and the salaries are still **strings** (`'3500'`). If a department appeared twice, `x + y` would concatenate the strings (`'30003000'`) instead of adding them — exactly the type problem the course warns about, and why `.cast("int")` should come before aggregating.

**Exercise — complex aggregations.** Average and total salaries of large US companies.

In [ ]:
# Average salaries at large us companies
large_companies = salaries_df.filter(salaries_df.company_size == "L").filter(salaries_df.company_location == "US").groupBy().avg("salary_in_usd")

# set a large companies variable for other analytics
large_companies = salaries_df.filter(salaries_df.company_size == "L").filter(salaries_df.company_location == "US")

# Total salaries in usd
large_companies.groupBy().sum("salary_in_usd").show()

![Total salary across large US companies](./screenshots/ch3_ex8_total_salary_large_us.png)

*Total salary across large US companies*

**Interpretation:** the total is ~194.3M USD. The first line computes the average but isn't displayed; reusing the filtered `large_companies` DataFrame is the cleaner pattern, since one filter feeds several aggregations.

### 3.4 — PySpark at scale: broadcast, execution plans and caching

- **Broadcast joins** send a small table to every worker, so the large table doesn't have to be shuffled across the cluster.
- **`explain()`** prints the physical execution plan — the way to see what Spark will actually do.
- **Caching** keeps a DataFrame in memory for fast reuse; **persisting** lets me choose the storage level (e.g. memory and disk) for larger data. Both should be released with `unpersist()` when done.
- **Avoid repeated actions** on the same data: each action recomputes the lineage unless the data is cached.

In [ ]:
from pyspark.sql.functions import broadcast
from pyspark import StorageLevel

joined_df = large_df.join(broadcast(small_df), on="key_column", how="inner")

df.filter(df.Age > 40).select("Name").explain()

df.cache()                                    # memory, for reuse
df.persist(StorageLevel.MEMORY_AND_DISK)      # explicit storage level
result = df.groupBy("column3").agg({"column4": "sum"})
df.unpersist()                                # free the memory

**Exercise — bringing it all together I.** Create a session, then build a DataFrame from the preloaded data and column definition.

In [ ]:
# Import SparkSession from pyspark.sql
from pyspark.sql import SparkSession

# Create my_spark
my_spark = SparkSession.builder.appName("final_spark").getOrCreate()

# Print my_spark
print(my_spark)

# Load dataset into a DataFrame
df = my_spark.createDataFrame(data, schema=columns)

df.show()

![Session confirmed and the employee DataFrame created](./screenshots/ch3_ex9_final_spark_dataframe.png)

*Session confirmed and the employee DataFrame created*

**Exercise — bringing it all together II.** Cache the DataFrame, aggregate, inspect the plan, then release the cache.

In [ ]:
# Cache the DataFrame
df.cache()

# Perform aggregation
agg_result = df.groupBy("Department").sum("Salary")
agg_result.show()

# Analyze the execution plan
agg_result.explain()

# Uncache the DataFrame
df.unpersist()

![Physical plan: the aggregation reads from InMemoryTableScan, confirming the cache is used](./screenshots/ch3_ex10_cache_explain_plan.png)

*Physical plan: the aggregation reads from InMemoryTableScan, confirming the cache is used*

**Interpretation (read bottom-up):**
1. `Scan ExistingRDD` — the source data.
2. `InMemoryRelation` / `InMemoryTableScan` — the cached copy is being read, so the cache is working.
3. `HashAggregate(partial_sum)` — each partition pre-aggregates locally.
4. `Exchange hashpartitioning(Department, 200)` — the **shuffle**: rows with the same department are moved to the same partition.
5. `HashAggregate(sum)` — the final per-department totals.

The partial-then-final aggregation is why Spark scales: most of the summing happens before any data crosses the network.

## 🧠 Key Takeaways

- **`SparkSession` is the entry point**, and `getOrCreate()` makes session creation safe to repeat.
- **Schemas are not a detail:** reading a CSV without `header` / schema silently produces `_c0` columns, string types and the header as a data row. Explicit `StructType` schemas give guaranteed types.
- **DataFrames are immutable:** every transformation returns a new DataFrame.
- **Join on clean keys:** rename keys to match, and choose the join type deliberately (left outer keeps all facts).
- **UDFs extend Spark with Python;** pandas UDFs are the vectorized, faster option for larger data.
- **Prefer DataFrames over RDDs** for structured data; RDDs are verbose and keep data types exactly as given (strings stay strings).
- **SQL and Python mix freely** through temp views, and `spark.sql()` returns a DataFrame.
- **Performance habits:** filter early, cast types before aggregating, cache reused data and unpersist it, use broadcast for small lookup tables, and read `explain()` plans for shuffles.

## 💼 Application in a Data Analyst / Data Engineer Role

- **ETL pipelines:** reading raw CSV/JSON with explicit schemas, cleaning nulls, deriving columns and joining fact and dimension data — the bronze → silver step in a Databricks lakehouse.
- **Analytics at scale:** the same `groupBy().agg()` and Spark SQL used here run unchanged on billions of rows in a Databricks cluster.
- **Data quality:** `describe()`, null handling and type casting are first checks before loading data into Delta tables.
- **Performance tuning:** recognizing shuffles in `explain()`, caching intermediate results and broadcasting lookups keeps cluster costs down.

## ✅ Skills Demonstrated

- Creating and reusing a `SparkSession`.
- Loading CSV and JSON into DataFrames with `header`, `inferSchema`, `sep` and explicit `StructType` schemas.
- Selecting, filtering, sorting, grouping and aggregating with the DataFrame API.
- Handling missing values (`na.drop`, `na.fill`, `isNotNull`) and creating, renaming and dropping columns.
- Joining DataFrames (including left outer joins on renamed keys), unions, and array / map / struct columns.
- Writing and registering PySpark UDFs and vectorized pandas UDFs.
- Converting DataFrames to RDDs and aggregating with `map()` and `reduceByKey()`.
- Querying DataFrames with Spark SQL through temporary views and summarizing with `describe()`.
- Using broadcast joins, `cache()` / `persist()` / `unpersist()`, and reading physical plans with `explain()`.